# 04 - Training an adapter, two ways

LoRA changes each weight matrix by a low-rank product; prefix tuning changes no weights at all and
instead learns keys and values that every attention layer reads ahead of the text. Both train here
with a hand-written backward pass checked against numerical differentiation, and both save in
PEFT's format - the adapters load in Python with `PeftModel.from_pretrained`.

*Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil.*

In [ ]:
#r "nuget: Gravicode.HFNet.GraviHub, 0.7.0"
#r "nuget: Gravicode.HFNet.GraviTokenizers, 0.7.0"
#r "nuget: Gravicode.HFNet.GraviDatasets, 0.7.0"
#r "nuget: Gravicode.HFNet.GraviTransformers, 0.7.0"
#r "nuget: Gravicode.HFNet.GraviPEFT, 0.7.0"

using Gravicode.HFNet.GraviDatasets;
using Gravicode.HFNet.GraviTransformers;
using Gravicode.HFNet.GraviPEFT;

In [ ]:
// Movie reviews, labelled 0 (negative) or 1 (positive). A slice keeps a CPU run to a minute or two.
var train = HubDatasets.Load("rotten_tomatoes", "train").Shuffle(seed: 1).Slice(0, 600);
var test = HubDatasets.Load("rotten_tomatoes", "test").Shuffle(seed: 1).Slice(0, 200);

string Name(double label) => label == 1 ? "positive" : "negative";
var texts = train.TextColumn("text");
var labels = train.NumericColumn("label").Select(Name).ToArray();
var heldOut = test.TextColumn("text");
var heldOutLabels = test.NumericColumn("label").Select(Name).ToArray();

using var model = TransformerModel.Load("google/bert_uncased_L-2_H-128_A-2");

## 1. LoRA

In [ ]:
var lora = PEFT.ApplyLoRA(model, new LoraConfig(Rank: 8, Alpha: 16));
Console.WriteLine(lora.Train(texts, labels, new TrainingOptions { Epochs = 3, BatchSize = 16, LearningRate = 2e-3 }));
Console.WriteLine($"held-out accuracy {lora.Score(heldOut, heldOutLabels):P1}");
lora.Predict("an absolute joy to watch")

## 2. Prefix tuning

In [ ]:
var prefix = PEFT.ApplyPrefixTuning(model, new PrefixTuningConfig(VirtualTokens: 10));
Console.WriteLine(prefix);   // how few parameters train

Console.WriteLine(prefix.Train(texts, labels, new TrainingOptions { Epochs = 3, BatchSize = 16, LearningRate = 2e-2 }));
Console.WriteLine($"held-out accuracy {prefix.Score(heldOut, heldOutLabels):P1}");
prefix.Predict("an absolute joy to watch")

## 3. Save, and load in Python

```python
from transformers import AutoModelForSequenceClassification
from peft import PeftModel
base = AutoModelForSequenceClassification.from_pretrained("google/bert_uncased_L-2_H-128_A-2", num_labels=2)
model = PeftModel.from_pretrained(base, "prefix-adapter")
```

One detail makes that work: when PEFT passes a prefix to BERT as `past_key_values`, BERT numbers
its positions from the prefix's length, so the first real token sits at position 10, not 0. HF.Net
does the same, which is why the logits agree to 1e-9.

In [ ]:
prefix.Save("prefix-adapter");
lora.SaveAdapter("lora-adapter");
System.IO.Directory.GetFiles("prefix-adapter")